# Datasets

In [ ]:
https://huggingface.co/datasets/Voxel51/scanned_receipts
https://huggingface.co/datasets/Hemgg/invoices-and-receipts_ocr_v2
https://huggingface.co/datasets/mahmoud2019/ReceiptQA
https://huggingface.co/datasets/amaye15/receipts
https://huggingface.co/datasets/shubh303/Invoice-to-Json

Vendor_name
Invoice_id
Purchase_date
Total
Subtotal
Tax
Items


Category= To be decided.


https://huggingface.co/datasets/mychen76/invoices-and-receipts_ocr_v1/    
https://huggingface.co/datasets/mychen76/invoices-and-receipts_ocr_v2/
https://huggingface.co/datasets/katanaml-org/invoices-donut-data-v1/ (same)



https://huggingface.co/datasets/Sigurdur/neytandinn-receipts/   "Marked"    Size: 1760
Missing:
- Subtotal
- Tax
- Invoice_id



https://huggingface.co/datasets/AjitRawat/invoice/    "Marked"    Size: 22


https://huggingface.co/datasets/naver-clova-ix/cord-v2






In [ ]:
https://huggingface.co/datasets/mychen76/invoices-and-receipts_ocr_v1/    
https://huggingface.co/datasets/mychen76/invoices-and-receipts_ocr_v2/

Tamil – Appen / Kaggle “Tamil language file type OCR image data”
Contains a RECEIPT category; about 356 receipt images across languages, with a Tamil subset.
Good source for Tamil receipts (images).
URL: Kaggle dataset by Appen 


Tamil – DataoceanAI “Tamil OCR Image Corpus”
1,493 total printed Tamil images, 11 categories including “receipts”.
Receipts are not counted separately, but you can safely expect dozens to low hundreds of receipt‑like images.
URL: DataoceanAI Tamil OCR dataset 


Tamil – FutureBeeAI printed OCR dataset
Includes Tamil invoices (not just receipts), among other document types, giving you more Indian‑style billing documents in Tamil 


Marathi – Roboflow “Mahavitran” Electricity Bills
37 electricity bill images, predominantly Marathi with some English 
Not receipts, but real billing documents in an Indian language.

In [ ]:
https://discuss.huggingface.co/t/seeking-indic-document-dataset-india-invoices-receipts-utility-bills-payment-advices-packing-lists-commercial-invoices-credit-notes/177055/6

In [ ]:
https://huggingface.co/AgamiAI/datasets     (Bank Statements)

In [ ]:
https://www.kaggle.com/datasets/ghassenkhaled/invoices-data


In [ ]:
https://www.shaip.com/solutions/optical-character-recognition-ocr-machine-learning/

In [ ]:
https://busy.in/invoice-format/
https://busy.in/invoice-format/restaurants/
https://busy.in/invoice-format/retail/
https://vyaparapp.in/invoice-formats#templates
https://mybillbook.in/s/invoice-format/

# Model Training

In [1]:
# Cell 1: Install Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets pillow json5

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-965p241b/unsloth_2b6e6a101f9a42bf9d49d5437eba03d4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-965p241b/unsloth_2b6e6a101f9a42bf9d49d5437eba03d4
  Resolved https://github.com/unslothai/unsloth.git to commit b5b52cfa48a74db5743c95be72fd3abee9f9b9a6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.3 MB/

In [2]:
import os
import gc
import torch
import json
import ast
from datasets import load_dataset, concatenate_datasets
from unsloth import FastVisionModel
from trl import SFTTrainer
from transformers import TrainingArguments
from PIL import Image

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
BASE_MODEL = "unsloth/Qwen2.5-VL-3B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen25vl_invoice_model"
MERGED_DIR = "/kaggle/working/qwen25vl_invoice_model/merged_hf"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

In [ ]:
# ---------------------------------------------------------------------------
# 0. CONFIG
# ---------------------------------------------------------------------------
BASE_MODEL = "unsloth/Qwen2.5-VL-3B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen25vl_invoice_model"
MERGED_DIR = "/kaggle/working/qwen25vl_invoice_model/merged_hf"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
 
# --- Kaggle 12hr-session speed levers, in order of impact -------------------
FINETUNE_VISION_LAYERS = False   # #1 lever. Vision-layer backprop is by far the most
                                  # expensive part of training. Qwen2.5-VL's vision encoder
                                  # is already strong at OCR; freezing it and only training
                                  # language layers is usually enough to learn the output
                                  # JSON schema, at 3-5x lower cost. Flip to True for a
                                  # quality-focused second pass once the pipeline is proven.
 
MAX_IMAGE_SIDE = 768              # #2 lever. Resize longest side before training -- vision
                                   # token count (and cost) scales with resolution.
 
TRAIN_SUBSET_SIZE = 3000          # #3 lever. Cap total training examples (None = use all).
                                   # ~9.7k combined examples x 2 epochs is what produced the
                                   # 12h+ ETA. Start small, confirm it converges + produces
                                   # sane JSON, then scale up in a later session/quota.
 
USE_GRAD_CHECKPOINT = True        # "unsloth" mode trades ~20-30% speed for less VRAM. With
                                   # vision layers frozen + a 3B model in 4bit on a 16GB T4
                                   # you likely have headroom -- try False first for a free
                                   # speed bump, fall back to True only if you OOM.
 
MAX_TRAIN_STEPS = 600             # hard cap regardless of dataset size -- gives you a known
                                   # wall-clock ceiling instead of trusting an epoch estimate.
SAVE_STEPS = 100                  # checkpoint often so a killed session is resumable.
RESUME_FROM_CHECKPOINT = False    # set to a checkpoint path on a second Kaggle session
                                   # to continue, e.g. "qwen25vl-3b-invoice-lora/checkpoint-300"

In [7]:

from datasets import Dataset
DATASET_PATH = "/kaggle/input/datasets/kumarmayank2177/processed-dataset/New data"
print("🔄 Loading dataset from disk...")
train_dataset = Dataset.load_from_disk(DATASET_PATH)
print(f"✅ Loaded {len(train_dataset)} examples")

🔄 Loading dataset from disk...


FileNotFoundError: No such files: '/kaggle/input/datasets/kumarmayank2177/processed-dataset/New data/dataset_info.json', nor '/kaggle/input/datasets/kumarmayank2177/processed-dataset/New data/state.json' found. Expected to load a `Dataset` object but provided path is not a `Dataset`.

In [8]:
# ==========================================
# 5. MODEL & LORA
# ==========================================
print("🔄 Loading Qwen2.5-VL 3B...")
model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    random_state=3407,
)

#FastVisionModel.for_training(model)

🔄 Loading Qwen2.5-VL 3B...
==((====))==  Unsloth 2026.7.6: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [12]:
import gc
import torch
from datasets import Dataset
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [15]:
# ==========================================
# 6. TRAIN (STABILITY FIXES)
# ==========================================
print("🚀 Starting training...")

trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,          # ✅ Lowered: safer for 4-bit vision
        gradient_accumulation_steps=8,          # ✅ Effective batch = 8
        warmup_steps=10,                       # ✅ % of steps, not fixed
        max_steps=900,
        learning_rate=1e-4,                    # ✅ Lowered further
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=2,
        optim="adamw_torch",                    # ✅ 8-bit optim can NaN with vision; full adamw is stabler
        weight_decay=0.01,
        lr_scheduler_type="linear",             # ✅ Linear is stabler than cosine near start
        seed=3407,
        output_dir="/kaggle/working/qwen25vl_invoice_modelcheckpoints",
        save_strategy="steps",
        save_steps=900,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ_LENGTH,
        max_grad_norm=0.3,                      # ✅ Aggressive clipping prevents NaN spikes                # ✅ Avoids length-sorting artifacts with images
    ),
)

trainer.train()

🚀 Starting training...
Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,886 | Num Epochs = 2 | Total steps = 900
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,784,556,544 (0.79% trained)


Step,Training Loss
2,0.758273
4,0.645680
6,0.612644
8,0.564985
10,0.590764
12,0.393727
14,0.406583
16,0.283294
18,0.270631
20,0.220943


KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------------------
# 4. SAVE — LoRA adapter, then merged full-precision model (needed for GGUF export)
# ---------------------------------------------------------------------------
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
processor.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")
 
# Merge LoRA into base weights -> standard HF safetensors model.
# This merged model is what step 2 (GGUF conversion) needs as input.
model.save_pretrained_merged(MERGED_DIR, processor, save_method="merged_16bit")
print(f"Merged model saved to {MERGED_DIR} — feed this into step 2 (GGUF conversion).")

# Model 1

In [1]:
# Cell 1: Install Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets pillow json5

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-2o4egve7/unsloth_2044ac100aec438f9840de950445544b
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-2o4egve7/unsloth_2044ac100aec438f9840de950445544b
  Resolved https://github.com/unslothai/unsloth.git to commit b41b819a4ec7579f130836e28a9b6f9bc5a403a5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.0 MB

In [2]:
import io
import gc
import json
import ast
from PIL import Image
import os
import torch
from datasets import load_from_disk
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# ==========================================
# CONFIG
# ==========================================
BASE_MODEL = "unsloth/Qwen2.5-VL-3B-Instruct"
DATASET_CACHE = "/kaggle/working/qwen25vl_invoice_model/processed_dataset"
OUTPUT_DIR = "/kaggle/working/qwen25vl_invoice_model/checkpoints"
MERGED_DIR = "/kaggle/working/qwen25vl_invoice_model/merged"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

In [4]:
# Kaggle T4 time-budget levers
FINETUNE_VISION_LAYERS = False   # biggest cost lever -- keep off unless you have quota to spare
MAX_TRAIN_STEPS = 900
SAVE_STEPS = 150
SAVE_TOTAL_LIMIT = 3
RESUME_FROM_CHECKPOINT = False   # set to a checkpoint path to continue a killed session
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
# ==========================================
# 1. LOAD THE ALREADY-PREPARED DATASET
# ==========================================
print("Loading prepared dataset...")
from datasets import Dataset
DATASET_PATH = "/kaggle/input/datasets/kumarmayank2177/processed-dataset/kaggle/working/qwen25vl_invoice_model/processed_dataset/final"
print("🔄 Loading dataset from disk...")
raw_dataset = Dataset.load_from_disk(DATASET_PATH)
print(f"✅ Loaded {len(train_dataset)} examples")

Loading prepared dataset...
🔄 Loading dataset from disk...
✅ Loaded 4868 examples


In [10]:
SYSTEM_PROMPT = (
    "You are an expert at reading invoice and receipt images. "
    "Extract all fields into a single well-formed JSON object. "
    "Do not invent values that are not visible in the image."
)
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [13]:

class ConversationDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, system_prompt):
        self.ds = hf_dataset
        self.system_prompt = system_prompt
 
    def __len__(self):
        return len(self.ds)
 
    def __getitem__(self, idx):
        example = self.ds[idx]  # flat columns -> image decodes to PIL.Image here, on demand
        return {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": example["image"]},
                        {"type": "text", "text": f"{self.system_prompt}\n\nExtract the structured JSON data from this document."},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": example["target_text"]}],
                },
            ]
        }
 
 
train_dataset = ConversationDataset(raw_dataset, SYSTEM_PROMPT)
print(f"✅ Loaded {len(train_dataset)} examples")

✅ Loaded 4868 examples


In [23]:
print(type(train_dataset))

from datasets import load_dataset

dataset= load_dataset(raw_dataset)

df= dataset.to_json(dataset)

print(df.head)

<class '__main__.ConversationDataset'>


TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'Dataset'

In [31]:
# ==========================================
# 2. LOAD MODEL + LORA
# ==========================================
model, processor = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=LOAD_IN_4BIT,
    use_gradient_checkpointing="unsloth",
)
 
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=FINETUNE_VISION_LAYERS,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,                    # ↓ or keep 16
    lora_alpha=16,
    lora_dropout=0.0,       # ↓ was 0.05
    bias="none",
    random_state=3407,
    use_rslora=True,        # ← was False
)
 

==((====))==  Unsloth 2026.8.4: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found


In [14]:
# ==========================================
# 3. PREFLIGHT CHECK
# ==========================================
def preflight_check(dataset, processor, n=8):
    print("\n--- Preflight: checking label token counts on a few examples ---")
    collator = UnslothVisionDataCollator(model, processor)
    sample = [dataset[i] for i in range(min(n, len(dataset)))]
    batch = collator(sample)
    labels = batch["labels"]
    bad = 0
    for i in range(labels.shape[0]):
        non_masked = (labels[i] != -100).sum().item()
        print(f"  example {i}: {non_masked} non-masked label tokens (seq len {labels.shape[1]})")
        if non_masked == 0:
            bad += 1
    if bad:
        print(f"WARNING: {bad}/{len(sample)} examples have ZERO non-masked labels.")
    else:
        print("OK -- all sampled examples have non-empty labels.\n")
    return bad == 0
 
assert preflight_check(train_dataset, processor), \
    "Preflight check failed -- fix the dataset before training (see warning above)."
 
# ==========================================
# 4. NaN GUARD
# ==========================================
class NaNGuardCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        loss = (logs or {}).get("loss")
        if loss is not None and (loss != loss):
            print(f"\nNaN loss detected at step {state.global_step}. Stopping.")
            control.should_training_stop = True
        return control
 


--- Preflight: checking label token counts on a few examples ---
Unsloth: Model does not have a default image size - using 512
  example 0: 480 non-masked label tokens (seq len 1349)
  example 1: 172 non-masked label tokens (seq len 1349)
  example 2: 879 non-masked label tokens (seq len 1349)
  example 3: 611 non-masked label tokens (seq len 1349)
  example 4: 744 non-masked label tokens (seq len 1349)
  example 5: 311 non-masked label tokens (seq len 1349)
  example 6: 245 non-masked label tokens (seq len 1349)
  example 7: 422 non-masked label tokens (seq len 1349)
OK -- all sampled examples have non-empty labels.



In [32]:
FastVisionModel.for_training(model)
 
trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    callbacks=[NaNGuardCallback()],
    args=SFTConfig(
        per_device_train_batch_size=1,          # ↓ safer for FP16
        gradient_accumulation_steps=4,            # effective batch = 4
        warmup_steps=max(20, int(0.05 * MAX_TRAIN_STEPS)),
        max_steps=MAX_TRAIN_STEPS,
        learning_rate=5e-5,                     # ↓ was 2e-4
        fp16=True,
        bf16=False,
        fp16_full_eval=False,                    # ← add
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        max_grad_norm=0.5,
        seed=3407,
        output_dir=OUTPUT_DIR,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=MAX_SEQ_LENGTH,
    ),
)
 
trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT or None)

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,868 | Num Epochs = 1 | Total steps = 900
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 14,966,784 of 3,769,589,760 (0.40% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.720625
20,nan



NaN loss detected at step 20. Stopping.


TrainOutput(global_step=20, training_loss=nan, metrics={'train_runtime': 219.8831, 'train_samples_per_second': 16.372, 'train_steps_per_second': 4.093, 'total_flos': 1376342413541376.0, 'train_loss': nan, 'epoch': 0.016433853738701727})

In [ ]:
 
# ==========================================
# 6. SAVE — LoRA adapter, then merged full-precision model for GGUF export
# ==========================================
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
processor.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")
 
model.save_pretrained_merged(MERGED_DIR, processor, save_method="merged_16bit")
print(f"Merged model saved to {MERGED_DIR} -- feed this into GGUF conversion next.")